# 04 — Prophet-STL (Custom) Modeling

**Warning:** This notebook does NOT use the Meta/Facebook `prophet` package. It runs a local, custom "Prophet‑style" implementation (`src/models/prophet_model.py`) that uses STL + Fourier + Ridge. All outputs from this notebook are labeled `Prophet-STL (Custom)` to avoid confusion with the official library.
This notebook runs the Prophet-style decomposition pipeline on all selected stocks using `src/models/prophet_model.py`.
It saves metrics, rolling predictions, and 5-day forecasts to the `outputs/` folders.

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
PROC_DIR = ROOT / 'data' / 'processed'
REPORT_DIR = ROOT / 'outputs' / 'reports'
PRED_DIR = ROOT / 'outputs' / 'predictions'
CHART_DIR = ROOT / 'outputs' / 'charts' / 'prophet'
for directory in [REPORT_DIR, PRED_DIR, CHART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
from src.data.preprocessor import SELECTED, NAMES
from src.models.prophet_model import ProphetStyleForecaster

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

def load_close_series(file_path: Path) -> pd.Series:
    frame = pd.read_csv(file_path, index_col=0, parse_dates=True)
    if 'Close' not in frame.columns:
        raise ValueError(f'Missing Close column in {file_path}')
    series = frame['Close'].dropna().sort_index()
    series.name = file_path.stem
    return series

metrics_rows = []
forecast_rows = []
actual_rows = []

for ticker in SELECTED:
    try:
        train_path = PROC_DIR / f'{ticker}_train_full.csv'
        test_path = PROC_DIR / f'{ticker}_test_full.csv'
        if not train_path.exists() or not test_path.exists():
            print(f'Skipping {ticker}: processed files not found')
            continue

        train_series = load_close_series(train_path)
        test_series = load_close_series(test_path)

        model = ProphetStyleForecaster(ticker=ticker, name=NAMES.get(ticker, ticker))
        model.run(train_series, test_series)

        metrics_rows.append(model.metrics)
        forecast_rows.append(model.forecast_5d.reset_index())
        actual_rows.append(pd.DataFrame({
            'Date': test_series.index,
            'Ticker': ticker,
            'Company': NAMES.get(ticker, ticker),
            'Actual': test_series.values,
            'Predicted': model.predictions.values,
            'Model': model.metrics['Model'],
        }))
        print(f"{ticker}: RMSE={model.metrics['RMSE']:.4f}  DA={model.metrics['DA_%']:.2f}%")
    except Exception as e:
        print(f"ERROR — {ticker}: {type(e).__name__}: {e}")
        continue

metrics_df = pd.DataFrame(metrics_rows).sort_values(['RMSE', 'MAE']).reset_index(drop=True)
forecasts_df = pd.concat(forecast_rows, ignore_index=True) if forecast_rows else pd.DataFrame()
actual_df = pd.concat(actual_rows, ignore_index=True) if actual_rows else pd.DataFrame()

metrics_df.to_csv(REPORT_DIR / 'prophet_metrics.csv', index=False)
forecasts_df.to_csv(PRED_DIR / 'prophet_5day_forecasts.csv', index=False)
actual_df.to_csv(PRED_DIR / 'prophet_actual_vs_predicted.csv', index=False)

if not metrics_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    plot_df = metrics_df[['Ticker', 'RMSE']].sort_values('RMSE')
    ax.bar(plot_df['Ticker'], plot_df['RMSE'], color='#2ca02c')
    ax.set_title('Prophet-STL (Custom) RMSE by Stock')
    ax.set_ylabel('RMSE')
    ax.set_xlabel('Ticker')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(CHART_DIR / 'prophet_rmse_by_ticker.png', dpi=150, bbox_inches='tight')
    plt.show()

display(metrics_df.head(10))
print(f'Saved: {REPORT_DIR / "prophet_metrics.csv"}')
print(f'Saved: {PRED_DIR / "prophet_5day_forecasts.csv"}')
print(f'Saved: {PRED_DIR / "prophet_actual_vs_predicted.csv"}')